# Training loop for mini-gLM

## Load data

Import packages and configure root.

In [ ]:
%pip install pyfaidx
%pip install wandb

import numpy as np
import pandas as pd
import sys, subprocess
import torch
import torch.nn as nn
import torch.nn.functional as F
import wandb

from pathlib import Path
from huggingface_hub import hf_hub_download
from src.transformer import MoETransformer
from torch.utils.data import DataLoader

# Configure root
COLAB = Path("/content").exists()
repo_url = "https://github.com/eddykang06/mini-gLM.git"
repo_dir = Path("mini-gLM")
if COLAB:
    root = Path("/content/mini-gLM")
    if not repo_dir.exists():
        subprocess.run(["git", "clone", repo_url])
else:
    root = Path.cwd().parent
sys.path.insert(0, str(root))

# Use GPU if available
torch.manual_seed(111)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## Training

Initialize wandb run.

In [ ]:
run = wandb.init(
    entity = "eddykang06-harvard-university",
    project = "glm",

    # Store hyperparameters
    config = {

    }
)

Configure the MLM training loop.
- Include W&B experiment tracking
- Easy parameter loading
- Mixed precision tuning
- Muon optimizer
- Figure out how to move the attention mask into the model?

In [ ]:
from src.data import hg38Data, DynamicBatchSampler, MLMCollator
from src.model import DenseGLM

def train_glm(epochs, lr, model_params, train_sampler, val_sampler, collate_fn, train_dataset, val_dataset, aux_loss_coef):

    # Load train and val data
    train_loader = DataLoader(
        dataset = train_dataset, 
        batch_sampler = train_sampler,
        collate_fn = collate_fn      
    )
    val_loader = DataLoader(
        dataset = val_dataset,
        batch_sampler = val_sampler,
        collate_fn = collate_fn
    )

    model = DenseGLM(**model_params)
    model.to(device).float()

    # Muon optimizer
    optim = torch.optim.Muon(model.parameters(), lr = lr)

    train_losses = []
    val_losses = []

    for epoch in range(epochs):

        # Train loss
        epoch_train_loss = 0.0

        for batch_items in train_loader:
        
            # Store batch items (match with output of collate_fn)
            batch = batch_items["batch"].to(device).float()
            labels = batch_items["labels"].to(device).float()
            predict_mask = batch_items["predict_mask"].to(device)
            attention_mask = batch_items["attention_mask"].to(device)

            # Generate predicted tokens and apply prediction mask to labels
            logits, aux_loss = model(batch, attention_mask)
            labels[~predict_mask] = -100

            # Initiate CE loss
            loss_fn = nn.CrossEntropyLoss(ignore_index = -100)

            # Calculate CE loss
            loss = loss_fn(
                logits.view(-1, logits.shape(-1))  # [B*L, vocab_size]
                labels.view(-1)                    # [B*L]
            )

            # Add scaled auxiliary MoE loss
            loss += aux_loss * aux_loss_coef

            # clear gradient, backprop, update params
            optim.zero_grad(); loss.backward(); optim.step()

            # Total loss = average batch loss * batch size
            epoch_train_loss += loss.item() * logits.size(0)
        
        # Compute average
        train_losses.append(epoch_train_loss / len(train_dataset))
    
        epoch_val_loss = 0.0

        for batch_items in val_loader:

            with torch.no_grad():

                # Store batch items 
                batch = batch_items["batch"].to(device).float()
                labels = batch_items["labels"].to(device).float()
                predict_mask = batch_items["predict_mask"].to(device)
                attention_mask = batch_items["attention_mask"].to(device)

                # Generate predicted tokens and apply prediction mask to labels (ignore aux loss)
                logits, _ = model(batch, attention_mask)
                labels[~predict_mask] = -100

                # Initiate CE loss
                loss_fn = nn.CrossEntropyLoss(ignore_index = -100)

                # Calculate CE loss
                loss = loss_fn(
                    logits.view(-1, logits.shape(-1)),  # [B*L, vocab_size]
                    labels.view(-1)                    # [B*L]
                )

                # Total loss = average batch loss * batch size
                epoch_val_loss += loss.item() * logits.size(0)

        # Copmpute average loss
        val_losses.append(epoch_val_loss / len(val_dataset))
    
    # Log losses to WandB
    run.log({
        "test_loss": train_losses[-1],
        "val_loss": val_losses[-1]
    })
    
    print("Training complete!")

    return model

Configure parameters.

In [ ]:
# Get data and tokenizer paths from HF
train_path = hf_hub_download(
    repo_id = "eddykang06/mini-gLM-data",
    repo_type = "dataset",
    filename = "pretraining-train.csv.gz",
)
val_path = hf_hub_download(
    repo_id = "eddykang06/mini-gLM-data",
    repo_type = "dataset",
    filename = "pretraining-train.csv.gz",
)
merge_rules_path = hf_hub_download(
    repo_id = "eddykang06/mini-gLM-data",
    repo_type = "dataset",
    filename = "merge-rules-512.json",
)
token_map_path = hf_hub_download(
    repo_id = "eddykang06/mini-gLM-data",
    repo_type = "dataset",
    filename = "merge-rules-512.json",
) 

# Get data
train_dataset = hg38Data(
    data_path = train_path,
    merge_rules_path = merge_rules_path,
    token_map_path = token_map_path
)
val_dataset = hg38Data(
    data_path = val_path,
    merge_rules_path = merge_rules_path,
    token_map_path = token_map_path
)

# Define batch sampler and collator
train_sampler = DynamicBatchSampler(
    dataset = train_dataset,
    batch_space = 
)
val_sampler = DynamicBatchSampler(
    dataset = val_dataset,
    batch_space = 

)
collate_fn = MLMCollator(
    vocab_size = 512,
    predict_prob = 0.15,
    masking_prob = 0.80,
    randomize_prob = 0.10
)

# Define model params
model_params = {
    "vocab_size": 512,
    "num_blocks": 5, 
    "d_model": 384,     
    "num_heads": 8,  
    "h_dim": 1024,      
    "num_experts": 8,
    "top_k": 2,
    "p_drop": 0.1
}

Train.

In [ ]:
# 30 epochs to start?
model = train_glm(
    epochs = 30, 
    lr = 0.0001, 
    model_params = model_params, 
    train_sampler = train_sampler,
    val_sampler = val_sampler,
    collate_fn = collate_fn, 
    train_dataset = train_dataset, 
    val_dataset = val_dataset
)
